In [35]:
import pandas as pd
import numpy as np

# ---------------- LOAD ---------------- #

sasa = pd.read_csv("/Users/varadakhot/Library/CloudStorage/OneDrive-Friedrich-Schiller-UniversitätJena/MCP_struct/reference_pdbs/sasa_analysis/sasa_capsid_estimated_all_proteins.tsv", sep="\t")

sasa['accession'] = np.where(sasa['protein'].str.contains("EC"),"EC6098_reference",sasa['accession'])
sasa['msa_col'] = np.where(sasa['protein'].str.contains("EC"),sasa['local_pos'],sasa['msa_col'])

interactions = pd.read_csv("/Users/varadakhot/Library/CloudStorage/OneDrive-Friedrich-Schiller-UniversitätJena/MCP_struct/reference_pdbs/new_comparisons/RF/feature_imp_with_interactions_allproteins.tsv", sep="\t")


In [36]:
print(interactions)

                 protein ecosystem_subtype  A_msa_col biochemical_interaction  \
0       EC6098_reference                 0          3               cation-pi   
1       EC6098_reference                 0          3               cation-pi   
2       EC6098_reference                 0          3             hydrophobic   
3       EC6098_reference                 0        377             hydrophobic   
4       EC6098_reference                 0        378             hydrophobic   
...                  ...               ...        ...                     ...   
716639  MGYP003333944615           Oceanic        100         non-interacting   
716640  MGYP003333944615           Oceanic        105         non-interacting   
716641  MGYP003333944615           Oceanic         21         non-interacting   
716642  MGYP003333944615           Oceanic        152         non-interacting   
716643  MGYP003333944615           Oceanic        425         non-interacting   

            interaction  fe

In [37]:
sasa.loc[sasa['protein'].str.contains("EC")]

,protein,accession,local_pos,AA,sasa_monomer,msa_col,ec6098_pos,reduction_factor,sasa_capsid_est
0,EC6098_reference_unrelaxed_rank_001_alphafold2...,EC6098_reference,1,M,256.9207,1.0,NaN,NaN,NaN
1,EC6098_reference_unrelaxed_rank_001_alphafold2...,EC6098_reference,2,S,110.9242,2.0,NaN,NaN,NaN
2,EC6098_reference_unrelaxed_rank_001_alphafold2...,EC6098_reference,3,K,195.7245,3.0,NaN,NaN,NaN
3,EC6098_reference_unrelaxed_rank_001_alphafold2...,EC6098_reference,4,F,209.9670,4.0,NaN,NaN,NaN
4,EC6098_reference_unrelaxed_rank_001_alphafold2...,EC6098_reference,5,G,77.7775,5.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
561,EC6098_reference_unrelaxed_rank_001_alphafold2...,EC6098_reference,562,L,127.1450,562.0,NaN,NaN,NaN
562,EC6098_reference_unrelaxed_rank_001_alphafold2...,EC6098_reference,563,I,178.9449,563.0,NaN,NaN,NaN
563,EC6098_reference_unrelaxed_rank_001_alphafold2...,EC6098_reference,564,D,107.5669,564.0,NaN,NaN,NaN
564,EC6098_reference_unrelaxed_rank_001_alphafold2...,EC6098_reference,565,H,131.8778,565.0,NaN,NaN,NaN


In [38]:
# msa_col comes in as float due to NaNs mixed in elsewhere in the column
sasa_mapped = sasa.dropna(subset=["msa_col"]).copy()
sasa_mapped["msa_col"] = sasa_mapped["msa_col"].astype(int)

# ---------------- EXTRACT ACCESSION FROM interactions['protein'] ---------------- #

interactions["accession"] = interactions["protein"].str.split("|").str[0]

# ---------------- MERGE ---------------- #

combined = interactions.merge(
    sasa_mapped[["accession", "msa_col", "AA", "local_pos", "sasa_monomer",
                 "reduction_factor", "sasa_capsid_est"]],
    left_on=["accession", "A_msa_col"],
    right_on=["accession", "msa_col"],
    how="left"
).drop(columns=["msa_col"])

# sanity check
missing = combined[combined["sasa_monomer"].isna()]
print(f"{len(missing)} / {len(combined)} interaction rows have no matching SASA")
if len(missing) > 0:
    print("Sample unmatched accessions:", missing["accession"].unique()[:5])

combined.to_csv("combined_interactions_sasa.tsv", sep="\t", index=False)

75587 / 742268 interaction rows have no matching SASA
Sample unmatched accessions: ['IMGVR_UViG_3300027969_000035' 'IMGVR_UViG_3300027969_000036'
 'IMGVR_UViG_3300027973_000026' 'IMGVR_UViG_3300027974_000782'
 'IMGVR_UViG_3300028553_000004']


In [39]:
combined

,protein,ecosystem_subtype,A_msa_col,biochemical_interaction,interaction,feature_importance_vals_A,biome_predictor_A,accession,AA,local_pos,sasa_monomer,reduction_factor,sasa_capsid_est
0,EC6098_reference,0,3,cation-pi,self,0.000261,ocean,EC6098_reference,K,3.0,195.7245,NaN,NaN
1,EC6098_reference,0,3,cation-pi,inter_capsomer,0.000261,ocean,EC6098_reference,K,3.0,195.7245,NaN,NaN
2,EC6098_reference,0,3,hydrophobic,inter_capsomer,0.000261,ocean,EC6098_reference,K,3.0,195.7245,NaN,NaN
3,EC6098_reference,0,377,hydrophobic,inter_capsomer,0.000668,lake,EC6098_reference,P,377.0,80.9929,NaN,NaN
4,EC6098_reference,0,378,hydrophobic,inter_capsomer,0.001001,lake,EC6098_reference,V,378.0,68.2479,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
742263,MGYP003333944615,Oceanic,100,non-interacting,non-interacting,0.000067,ocean,MGYP003333944615,G,87.0,17.4829,0.999934,17.481739
742264,MGYP003333944615,Oceanic,105,non-interacting,non-interacting,0.000065,ocean,MGYP003333944615,G,92.0,25.2904,0.238800,6.039343
742265,MGYP003333944615,Oceanic,21,non-interacting,non-interacting,0.000058,ocean,MGYP003333944615,K,8.0,169.1520,0.498290,84.286710
742266,MGYP003333944615,Oceanic,152,non-interacting,non-interacting,0.000040,ocean,MGYP003333944615,F,137.0,2.7815,1.000000,2.781500
